# Aquaplanet with a customized initial condition

This notebook does two things to the aquaplanet of `01_aquaplanet.ipynb`: it writes the coupling between the components out as one plain Python function, and it changes one field of the initial state before the run. It builds the SPEEDY T31L8 atmosphere over a slab ocean with slab sea ice, adds a smooth wave to the initial sea surface temperature, runs 30 days, and plots the final temperature and the area-mean temperature over time. On a laptop CPU it takes about 2 minutes.

The components are exactly the ones `01_aquaplanet.ipynb` builds. That notebook takes its wiring from `default_exchangers`, the standard exchange table; this one writes the same exchange by hand, so that everything passing between the atmosphere, the ocean and the sea ice is on the page. Like that notebook, it is a teaching build with SPEEDY's reference parameters, not a calibrated climate; the calibrated model is `jem.configurations.load("aquaplanet-slab")`.

In [ ]:
from pathlib import Path

import jax.numpy as jnp

from jem import plot, replace_field, run_chunked

output_dir = (Path("output") / "01-02_customized_initial_condition").resolve()
output_dir.mkdir(parents=True, exist_ok=True)

## Build the model, with the exchange written by hand

An exchanger is the only place where components exchange information. It is a function that takes every component's carry (a dict with the sections `state`, `forcing`, `derived` and `params`) and the coupling time, and returns the carries to continue with. The one below moves four fields:

| From | To | Why |
| --- | --- | --- |
| `atm.derived.sea_total_heat_flux` | `ocn.forcing.total_heat_flux` | The net surface heat flux over the sea part of each cell, positive upward, is what warms or cools the mixed layer. |
| `ocn.state.sea_surface_temperature` | `atm.forcing.sea_surface_temperature` | The ocean's temperature is the atmosphere's lower boundary condition. |
| `ocn.state.sea_surface_temperature` | `seaice.forcing.sea_surface_temperature` | The slab sea ice is diagnosed from the sea surface temperature alone. |
| `seaice.derived.ice_fraction` | `atm.forcing.sice_am` | The ice cover is what the atmosphere's surface-flux scheme sees. |

Coupling is lagged by one coupled step: the exchanger runs first, so each component steps with what the others held at the end of the previous step.

A hand-written exchanger has to keep two rules, because the coupled step runs inside `jax.lax.scan`:

- **Build new structs, never assign into the ones passed in.** `.replace(...)` returns a new forcing struct and `dict(carry, forcing=...)` a new carry. Assigning in place would overwrite the caller's initial carry when a step is run outside `jit`.
- **Return the structure that came in.** The same components, sections and fields, with the same shapes and dtypes. The coupler checks the structure after every workflow element and names the one that changed it; `lax.scan` rejects a changed shape or dtype.

`default_exchangers(components)` builds these same four rows as a `jem.exchangers.Exchange` table, and `tests/unit/test_notebook_equivalence.py` checks that the function below and that table give the same run. A table can be inspected and a function cannot, so a table is the better choice when the exchange only moves fields: an atmosphere whose boundary conditions come from a file has to be told which of them a function overwrites (`JCMComponent.set_exchanged_forcing`), where a table is read for it. This atmosphere's boundary conditions are plain fields, so nothing more is needed here. See `docs/source/design/exchange.md` for both forms.

In [ ]:
import jax_datetime as jdt
import jcm
from jcm.physics.speedy.speedy_coords import get_speedy_coords

from jem import Coupler
from jem.components import JCMComponent, SlabOceanModel, SlabSeaiceModel
from jem.components.slab import SlabGrid

start_date = jdt.to_datetime("2000-01-01")
coupling_timestep = jdt.to_timedelta(1, "day")

# The components of `01_aquaplanet.ipynb`: a plain jcm.model.Model wrapped as
# a component, and a slab ocean and slab sea ice on the atmosphere's own
# horizontal grid. With no fractional mask every cell is ocean.
atm_model = jcm.model.Model(coords=get_speedy_coords(), start_time=start_date)
grid = SlabGrid.from_coords(atm_model.coords.horizontal)
components = {
    "atm": JCMComponent(atm_model),
    "ocn": SlabOceanModel(grid),
    "seaice": SlabSeaiceModel(grid, name="seaice"),
}


def exchange(carries, time):
    """Move the coupled fields between the atmosphere, the ocean and the sea ice."""
    del time  # this exchange does not depend on the date
    atm, ocn, seaice = carries["atm"], carries["ocn"], carries["seaice"]

    # Read every source before writing anything, so the exchange is
    # simultaneous: no component is handed a field another row has already
    # replaced in this same call.
    sea_heat_flux = atm["derived"].sea_total_heat_flux
    sea_surface_temperature = ocn["state"].sea_surface_temperature
    ice_fraction = seaice["derived"].ice_fraction

    # Write each destination into a new forcing struct inside a new carry.
    atm = dict(atm, forcing=atm["forcing"].replace(
        sea_surface_temperature=sea_surface_temperature,
        sice_am=ice_fraction,
    ))
    ocn = dict(ocn, forcing=ocn["forcing"].replace(
        total_heat_flux=sea_heat_flux,
    ))
    seaice = dict(seaice, forcing=seaice["forcing"].replace(
        sea_surface_temperature=sea_surface_temperature,
    ))
    return dict(carries, atm=atm, ocn=ocn, seaice=seaice)


# The workflow is written out too: the exchange, then each component. It is
# the order the coupler would choose by default (every exchanger, then every
# component).
coupler = Coupler(
    components,
    {"exchange": exchange},
    workflow=["exchange", "atm", "ocn", "seaice"],
    coupling_timestep=coupling_timestep,
    start_date=start_date,
)
coupler

## Change one initial condition

`Coupler.initialize()` returns a `CoupledCarry`; `replace_field` returns a new one with a single `"component.section.field"` replaced, so the original is untouched.

In [ ]:
carry = coupler.initialize()
grid = coupler.components["ocn"].grid
bump = 5 * jnp.sin(2 * grid.longitude_radian) * jnp.cos(grid.latitude_radian) ** 3
sst = carry.components["ocn"]["state"].sea_surface_temperature
carry = replace_field(carry, "ocn.state.sea_surface_temperature", sst + bump)

In [ ]:
import xarray as xr

from jem.components.slab.grid import to_degrees

# The aquaplanet's ocean is on the atmosphere's separable lon/lat grid, so the
# field carries 1-D `lon`/`lat` coordinates in degrees, the same ones every
# component's dataset uses (a curvilinear grid has 2-D `longitude`/`latitude`
# over `x`/`y` instead; `map_plot` draws both).
assert grid.is_separable
perturbed_sst = xr.DataArray(
    sst + bump - 273.15,
    dims=grid.dims,
    coords={
        "lon": to_degrees(grid.longitude_axis_radian),
        "lat": to_degrees(grid.latitude_axis_radian),
    },
    name="sea_surface_temperature",
)
plot.map_plot(perturbed_sst, title="Perturbed initial sea surface temperature [°C]")

## Run it

In [ ]:
result = run_chunked(
    coupler,
    total_time="30 days",
    chunk="30 days",
    initial_carry=carry,
    output_dir=str(output_dir),
    subsample=3,       # 10 records out of 30 coupled days
    checkpoint_path=None,
)

In [ ]:
import matplotlib.pyplot as plt

ocn = plot.open_output(output_dir, "ocn")

fig, ax = plt.subplots()
sst_final = ocn["sea_surface_temperature"].isel(time=-1) - 273.15
plot.map_plot(sst_final, ax=ax, title="Final sea surface temperature [°C]")

fig, ax = plt.subplots()
plot.area_mean(ocn["sea_surface_temperature"]).plot(ax=ax)
ax.set_ylabel("Area-mean SST [K]")